**Installation des dépendances**

In [69]:
!pip install -q sentence-transformers chromadb langchain langchain-community \
             langchain-core pyflakes
 
import os, json, ast, sys, time, re
from pathlib import Path
 
print("✅ Dépendances installées")
print(f"   Python : {sys.version}")

✅ Dépendances installées
   Python : 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]


**Vérification GPU**

In [70]:
import torch
 
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n{'='*50}")
print(f"  Device disponible : {device.upper()}")
if torch.cuda.is_available():
    print(f"  GPU : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"{'='*50}")


  Device disponible : CUDA
  GPU : Tesla T4
  VRAM : 15.6 GB


In [71]:
import os

# Lister tous les fichiers disponibles
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/siwarbensalah/datapropre/flask_chunks.json


**Upload et extraction des données**

In [72]:
# ── CELLULE 3 : Chargement des données ───────────────────────────────────────

from pathlib import Path
import json

CHUNKS_FILE = Path("/kaggle/input/datasets/siwarbensalah/datapropre/flask_chunks.json")

print(f"Chargement : {CHUNKS_FILE}")

with open(CHUNKS_FILE, encoding="utf-8") as f:
    chunks = json.load(f)

print(f"✅ {len(chunks)} chunks chargés")

Chargement : /kaggle/input/datasets/siwarbensalah/datapropre/flask_chunks.json
✅ 388 chunks chargés


**Génération des embeddings sur GPU**

In [73]:
from sentence_transformers import SentenceTransformer
import numpy as np
 
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
CHROMA_DIR      = Path("/kaggle/working/vector_db")
COLLECTION_NAME = "flask_documentation"
 
print(f"\n{'='*50}")
print(f"  Chargement du modèle '{EMBEDDING_MODEL}'")
print(f"  Device : {device.upper()}")
print(f"{'='*50}")
 
model = SentenceTransformer(EMBEDDING_MODEL, device=device)
dim   = model.get_sentence_embedding_dimension()
print(f"✅ Modèle chargé — {dim} dimensions")
 
# Construire les textes à encoder
def build_text(chunk):
    parts = []
    if chunk.get("page_title"): parts.append(chunk["page_title"])
    if chunk.get("section"):    parts.append(chunk["section"])
    if chunk.get("text"):       parts.append(chunk["text"])
    return " | ".join(parts)
 
texts = [build_text(c) for c in chunks]
 
print(f"\nCalcul des embeddings pour {len(texts)} chunks...")
t0 = time.perf_counter()
 
vectors = model.encode(
    texts,
    batch_size  = 64 if device == "cuda" else 32,
    show_progress_bar   = True,
    convert_to_numpy    = True,
    normalize_embeddings= True,
)
 
elapsed = time.perf_counter() - t0
print(f"\n✅ Embeddings calculés en {elapsed:.2f}s ({elapsed/len(chunks)*1000:.1f} ms/chunk)")
print(f"   Shape : {vectors.shape}")
print(f"   Gain GPU estimé : ~{max(1, int(10 - elapsed/len(chunks)*10))}x plus rapide qu'en CPU")


  Chargement du modèle 'all-MiniLM-L6-v2'
  Device : CUDA


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Modèle chargé — 384 dimensions

Calcul des embeddings pour 388 chunks...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]


✅ Embeddings calculés en 0.88s (2.3 ms/chunk)
   Shape : (388, 384)
   Gain GPU estimé : ~9x plus rapide qu'en CPU


**Construction ChromaDB**

In [74]:
import chromadb
from chromadb.config import Settings
 
CHROMA_DIR.mkdir(parents=True, exist_ok=True)
 
client     = chromadb.PersistentClient(
    path=str(CHROMA_DIR),
    settings=Settings(anonymized_telemetry=False)
)
 
# Reset si déjà existant
try:
    client.delete_collection(COLLECTION_NAME)
    print("Collection existante supprimée")
except:
    pass
 
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)
 
# Insertion par lots
INSERT_BATCH = 100
inserted     = 0
 
for start in range(0, len(chunks), INSERT_BATCH):
    end   = min(start + INSERT_BATCH, len(chunks))
    batch = chunks[start:end]
    vecs  = vectors[start:end]
 
    metadatas = []
    for c in batch:
        metadatas.append({
            "source_url":  c.get("source_url", ""),
            "page_title":  c.get("page_title", ""),
            "section":     c.get("section", ""),
            "chunk_index": int(c.get("chunk_index", 0)),
            "has_code":    len(c.get("code_blocks", [])) > 0,
            "num_codes":   len(c.get("code_blocks", [])),
            "code_blocks": json.dumps(c.get("code_blocks", []), ensure_ascii=False),
        })
 
    collection.upsert(
        ids        = [c["chunk_id"] for c in batch],
        embeddings = vecs.tolist(),
        documents  = [c.get("text", "") for c in batch],
        metadatas  = metadatas,
    )
    inserted += len(batch)
    print(f"  {inserted}/{len(chunks)} insérés")
 
print(f"\n✅ ChromaDB construite : {collection.count()} documents")

Collection existante supprimée
  100/388 insérés
  200/388 insérés
  300/388 insérés
  388/388 insérés

✅ ChromaDB construite : 388 documents


**Retriever de base**

In [75]:
from dataclasses import dataclass, field
from typing import Optional
 
@dataclass
class RetrievalResult:
    """Un chunk retourné par la recherche sémantique."""
    chunk_id:    str
    text:        str
    page_title:  str
    section:     str
    source_url:  str
    score:       float
    code_blocks: list = field(default_factory=list)
    has_code:    bool = False
 
    def to_context_block(self) -> str:
        lines = [
            f"[SOURCE] {self.page_title} — {self.section}",
            f"[URL] {self.source_url}",
            f"[CONTENU]\n{self.text}",
        ]
        for block in self.code_blocks:
            lang = block.get("language", "")
            code = block.get("code", "")
            if code:
                lines.append(f"[CODE {lang.upper()}]\n```{lang}\n{code}\n```")
        return "\n".join(lines)
 
 
class BaseRetriever:
    """Retriever sémantique de base sur ChromaDB."""
 
    def __init__(self, embedding_model, collection, device="cpu"):
        self.model      = embedding_model
        self.collection = collection
        self.device     = device
 
    def search(self, query: str, top_k: int = 4, min_score: float = 0.25) -> list:
        if not query.strip():
            return []
 
        vector = self.model.encode(query, normalize_embeddings=True, device=self.device)
 
        raw = self.collection.query(
            query_embeddings=[vector.tolist()],
            n_results=min(top_k, self.collection.count()),
            include=["documents", "metadatas", "distances"],
        )
 
        results = []
        for cid, doc, meta, dist in zip(
            raw["ids"][0], raw["documents"][0],
            raw["metadatas"][0], raw["distances"][0]
        ):
            score = round(1.0 - dist / 2.0, 4)
            if score < min_score:
                continue
 
            code_blocks = []
            try:
                code_blocks = json.loads(meta.get("code_blocks", "[]"))
            except:
                pass
 
            results.append(RetrievalResult(
                chunk_id    = cid,
                text        = doc,
                page_title  = meta.get("page_title", ""),
                section     = meta.get("section", ""),
                source_url  = meta.get("source_url", ""),
                score       = score,
                code_blocks = code_blocks,
                has_code    = bool(meta.get("has_code", False)),
            ))
 
        return sorted(results, key=lambda x: x.score, reverse=True)
 
 
# Instancier le retriever de base
retriever = BaseRetriever(model, collection, device)
print("✅ BaseRetriever instancié")

✅ BaseRetriever instancié


**AGENT 1 — Reformulation de requête**

In [76]:
class QueryReformulationAgent:
    """
    Agent de reformulation de requête.
 
    Rôle : Si le retrieval initial retourne des résultats de mauvaise qualité
    (score < seuil ou 0 résultats), cet agent reformule la question en :
      1. Extrayant les concepts-clés Flask de la question
      2. Générant des variantes sémantiquement équivalentes
      3. Relançant la recherche avec la meilleure formulation
 
    Valeur ajoutée : améliore le taux de retrieval réussi pour les
    utilisateurs qui ne maîtrisent pas la terminologie Flask exacte.
    """
 
    # Synonymes et reformulations Flask courantes
    FLASK_SYNONYMS = {
        "route":        ["endpoint", "url rule", "decorator route", "path"],
        "blueprint":    ["module flask", "composant flask", "blueprint flask"],
        "debug":        ["mode debug", "debug mode", "débogage", "développement"],
        "template":     ["jinja", "jinja2", "html flask", "rendu"],
        "config":       ["configuration", "paramètres", "settings"],
        "session":      ["cookie session", "user session", "flask session"],
        "database":     ["db", "SQLAlchemy", "base de données", "ORM"],
        "error":        ["exception", "erreur http", "404", "500", "gestionnaire erreur"],
        "middleware":   ["before_request", "after_request", "hook"],
        "deployment":   ["production", "déploiement", "gunicorn", "wsgi"],
        "test":         ["unittest", "pytest", "test client", "test flask"],
        "request":      ["requête http", "form data", "json request"],
        "response":     ["réponse http", "return flask", "jsonify"],
        "login":        ["authentification", "auth", "flask-login", "connexion"],
        "static":       ["fichiers statiques", "css", "js", "assets"],
    }
 
    def __init__(self, retriever: BaseRetriever, min_score: float = 0.35):
        self.retriever = retriever
        self.min_score = min_score
 
    def _extract_key_concepts(self, query: str) -> list[str]:
        """Extrait les concepts Flask clés d'une question."""
        query_lower = query.lower()
        concepts    = []
        for concept in self.FLASK_SYNONYMS:
            if concept in query_lower:
                concepts.append(concept)
            for synonym in self.FLASK_SYNONYMS[concept]:
                if synonym in query_lower:
                    concepts.append(concept)
                    break
        return list(set(concepts))
 
    def _generate_reformulations(self, query: str, concepts: list[str]) -> list[str]:
        """Génère des variantes de la question pour améliorer le retrieval."""
        reformulations = [query]   # version originale toujours en premier
 
        # Reformulation 1 : ajouter "Flask" explicitement si absent
        if "flask" not in query.lower():
            reformulations.append(f"Flask {query}")
 
        # Reformulation 2 : traduire si question en français
        french_terms = {
            "comment": "how to",
            "créer":   "create",
            "définir": "define",
            "utiliser":"use",
            "activer": "enable",
            "configurer": "configure",
            "afficher": "display",
            "retourner": "return",
        }
        english_query = query
        for fr, en in french_terms.items():
            english_query = english_query.replace(fr, en)
        if english_query != query:
            reformulations.append(english_query)
 
        # Reformulation 3 : utiliser les synonymes Flask
        for concept in concepts:
            synonyms = self.FLASK_SYNONYMS.get(concept, [])
            for syn in synonyms[:2]:
                reformulations.append(query.replace(concept, syn))
 
        # Reformulation 4 : version technique directe
        if concepts:
            technical = f"Flask {' '.join(concepts)} documentation example"
            reformulations.append(technical)
 
        return reformulations[:5]  # maximum 5 tentatives
 
    def run(self, query: str, initial_results: list, top_k: int = 4) -> dict:
        """
        Exécute l'agent de reformulation.
 
        Paramètres :
          query           : question originale
          initial_results : résultats du retrieval initial
          top_k           : nombre de chunks à récupérer
 
        Retourne :
          {
            "reformulated": bool,          # True si reformulation effectuée
            "original_query": str,
            "used_query": str,             # formulation ayant donné les meilleurs résultats
            "results": list[RetrievalResult],
            "attempts": int,               # nombre de tentatives
            "best_score": float
          }
        """
        best_score = max((r.score for r in initial_results), default=0.0)
 
        # Si le retrieval initial est déjà satisfaisant
        if initial_results and best_score >= self.min_score:
            return {
                "reformulated": False,
                "original_query": query,
                "used_query":     query,
                "results":        initial_results,
                "attempts":       1,
                "best_score":     best_score,
            }
 
        print(f"  [Reformulation] Score initial faible ({best_score:.3f}) → tentative de reformulation")
 
        concepts        = self._extract_key_concepts(query)
        reformulations  = self._generate_reformulations(query, concepts)
 
        best_results    = initial_results
        best_query      = query
        best_score_seen = best_score
        attempts        = 1
 
        for variant in reformulations[1:]:  # ignorer la version originale déjà testée
            attempts += 1
            results  = self.retriever.search(variant, top_k=top_k, min_score=0.15)
 
            if results:
                current_best = max(r.score for r in results)
                if current_best > best_score_seen:
                    best_score_seen = current_best
                    best_results    = results
                    best_query      = variant
                    print(f"  [Reformulation] Meilleure variante trouvée (score={current_best:.3f}) : '{variant}'")
 
                    if current_best >= self.min_score:
                        break   # résultat suffisant — arrêter
 
        return {
            "reformulated": best_query != query,
            "original_query": query,
            "used_query":     best_query,
            "results":        best_results,
            "attempts":       attempts,
            "best_score":     best_score_seen,
        }
 

**AGENT 2 — Validation du code**

In [77]:
class CodeValidationAgent:
    """
    Agent de validation syntaxique du code Python.
 
    Rôle : Avant de retourner une réponse contenant du code, cet agent
    analyse les blocs de code Python extraits de la documentation et
    détecte les problèmes syntaxiques ou les usages dépréciés.
 
    Valeur ajoutée : garantit que le code fourni à l'utilisateur est
    au minimum syntaxiquement valide, réduisant le risque d'erreur.
    """
 
    # Patterns de code Flask invalides ou dépréciés en 3.0.x
    DEPRECATED_PATTERNS = [
        (r"app\.run\(debug=True\)",
         "Utilisez 'flask --debug run' en ligne de commande (Flask 3.0.x)"),
        (r"from flask\.ext\.",
         "Les extensions flask.ext sont obsolètes depuis Flask 1.0"),
        (r"flask\.helpers\.url_for",
         "Utilisez 'from flask import url_for' directement"),
        (r"@app\.before_first_request",
         "before_first_request est supprimé dans Flask 2.3+, utilisez with app.app_context()"),
    ]
 
    def validate_python(self, code: str) -> dict:
        """
        Valide syntaxiquement un bloc de code Python.
 
        Retourne :
          {
            "valid":    bool,
            "errors":   list[str],    # erreurs de syntaxe
            "warnings": list[str],    # usages dépréciés
            "fixed":    str | None    # tentative de correction automatique
          }
        """
        errors   = []
        warnings = []
        fixed    = None
 
        # Nettoyage du code
        code_clean = code.strip()
        if not code_clean:
            return {"valid": True, "errors": [], "warnings": [], "fixed": None}
 
        # Vérification syntaxique Python
        try:
            ast.parse(code_clean)
        except SyntaxError as e:
            errors.append(f"Erreur de syntaxe ligne {e.lineno} : {e.msg}")
 
            # Tentative de correction automatique simple
            fixed = self._try_fix(code_clean, e)
 
        # Vérification des patterns dépréciés
        for pattern, message in self.DEPRECATED_PATTERNS:
            if re.search(pattern, code_clean):
                warnings.append(f"⚠️ Usage déprécié : {message}")
 
        # Vérification indentation
        lines = code_clean.split("\n")
        for i, line in enumerate(lines):
            if line and not line[0].isspace() and line[0] != "#":
                pass  # ligne valide
            elif "\t" in line and "    " in line:
                warnings.append(f"Ligne {i+1} : mélange tabulations/espaces détecté")
 
        return {
            "valid":    len(errors) == 0,
            "errors":   errors,
            "warnings": warnings,
            "fixed":    fixed,
        }
 
    def _try_fix(self, code: str, error: SyntaxError) -> Optional[str]:
        """Tente une correction automatique simple."""
        lines = code.split("\n")
 
        # Fix 1 : parenthèse manquante en fin de fichier
        if "EOF" in str(error.msg) or "unexpected" in str(error.msg):
            open_p  = code.count("(")
            close_p = code.count(")")
            if open_p > close_p:
                return code + ")" * (open_p - close_p)
 
        # Fix 2 : deux-points manquants après def/if/for/with
        if error.lineno and error.lineno <= len(lines):
            line = lines[error.lineno - 1]
            if re.match(r"^(def |class |if |for |while |with )", line.strip()):
                if not line.rstrip().endswith(":"):
                    lines[error.lineno - 1] = line.rstrip() + ":"
                    return "\n".join(lines)
 
        return None
 
    def run(self, results: list[RetrievalResult]) -> dict:
        """
        Valide tous les blocs de code des résultats de retrieval.
 
        Retourne un rapport de validation complet.
        """
        validation_report = {
            "total_blocks": 0,
            "valid_blocks": 0,
            "blocks_with_errors":   [],
            "blocks_with_warnings": [],
            "all_valid": True,
        }
 
        for result in results:
            for block in result.code_blocks:
                if block.get("language", "") not in ("python", ""):
                    continue   # ne valider que le Python
 
                code   = block.get("code", "")
                report = self.validate_python(code)
 
                validation_report["total_blocks"] += 1
 
                if report["valid"]:
                    validation_report["valid_blocks"] += 1
                else:
                    validation_report["all_valid"] = False
                    validation_report["blocks_with_errors"].append({
                        "chunk_id": result.chunk_id,
                        "section":  result.section,
                        "errors":   report["errors"],
                        "fixed":    report["fixed"],
                        "original": code,
                    })
                    # Appliquer la correction si disponible
                    if report["fixed"]:
                        block["code"]     = report["fixed"]
                        block["corrected"]= True
 
                if report["warnings"]:
                    validation_report["blocks_with_warnings"].append({
                        "chunk_id": result.chunk_id,
                        "section":  result.section,
                        "warnings": report["warnings"],
                    })
 
        return validation_report
 

**AGENT 3 — Routing hors-scope**

In [78]:
class RoutingAgent:
    """
    Agent de routing et détection hors-scope.
 
    Rôle : Évalue la question AVANT le retrieval pour déterminer si elle
    est dans le périmètre de la documentation Flask. Les questions hors-scope
    sont interceptées et reçoivent un message adapté, sans solliciter
    inutilement le LLM.
 
    Stratégie de décision multicritère :
      1. Mots-clés Flask positifs → dans le scope
      2. Mots-clés hors-scope évidents → hors scope immédiat
      3. Score de retrieval trop bas → probablement hors scope
      4. Ambiguïté → laisser passer avec avertissement
 
    Valeur ajoutée : économise des ressources CPU/GPU, améliore l'expérience
    utilisateur en donnant des messages clairs plutôt que des réponses inventées.
    """
 
    # Indicateurs positifs : question probablement Flask
    FLASK_POSITIVE_KEYWORDS = {
        # Core Flask
        "flask", "route", "blueprint", "jinja", "werkzeug", "wsgi",
        "app.run", "app_context", "request", "response", "redirect",
        "url_for", "render_template", "jsonify", "make_response",
        "before_request", "after_request", "teardown",
        # Extensions communes
        "flask-login", "flask-sqlalchemy", "flask-wtf", "flask-migrate",
        "flask-cors", "flask-jwt",
        # Concepts
        "endpoint", "view function", "context", "g object", "session flask",
        "error handler", "signal", "cli flask", "testing flask",
        "déploiement flask", "configuration flask", "middleware flask",
    }
 
    # Indicateurs négatifs : question clairement hors-scope Flask
    OUT_OF_SCOPE_KEYWORDS = {
        # Autres frameworks web
        "django", "fastapi", "tornado", "aiohttp", "starlette", "express",
        "react", "vue", "angular", "svelte", "nextjs", "nuxt",
        # Machine learning (sauf si contexte Flask)
        "pytorch", "tensorflow", "keras", "scikit-learn", "pandas dataframe",
        "numpy array", "deep learning model", "neural network training",
        # Autres domaines
        "cuisine", "recette", "football", "sport", "météo", "film", "musique",
        "histoire", "géographie", "mathématiques", "physique", "chimie",
        # Sujets généraux non techniques
        "qui est", "quelle est la capitale", "combien coûte",
        "traduction", "définition de",
    }
 
    # Réponses hors-scope par catégorie
    OUT_OF_SCOPE_RESPONSES = {
        "other_framework": (
            "Cette question concerne un autre framework que Flask. "
            "Notre système est spécialisé exclusivement dans la documentation officielle "
            "Flask 3.0.x. Pour des questions sur {framework}, nous vous recommandons "
            "de consulter sa documentation officielle."
        ),
        "non_technical": (
            "Cette question ne semble pas concerner le développement avec Flask. "
            "Notre assistant est dédié au support technique de la documentation "
            "Flask 3.0.x. N'hésitez pas à poser des questions sur les routes, "
            "les blueprints, les templates, la configuration ou tout autre aspect de Flask."
        ),
        "low_relevance": (
            "Je ne trouve pas d'information suffisamment pertinente dans la "
            "documentation Flask pour répondre à cette question. "
            "Cela peut signifier que le sujet n'est pas couvert par Flask 3.0.x "
            "ou que la question nécessite une reformulation plus spécifique. "
            "Essayez d'utiliser des termes Flask précis dans votre question."
        ),
    }
 
    def __init__(self, retriever: BaseRetriever,
                 low_relevance_threshold: float = 0.30):
        self.retriever  = retriever
        self.threshold  = low_relevance_threshold
 
    def _detect_other_framework(self, query: str) -> Optional[str]:
        """Détecte si la question concerne un autre framework spécifique."""
        query_lower = query.lower()
        frameworks  = ["django", "fastapi", "tornado", "aiohttp", "react",
                       "vue", "angular", "express", "nextjs"]
        for fw in frameworks:
            if fw in query_lower:
                return fw
        return None
 
    def _compute_flask_relevance_score(self, query: str) -> float:
        """
        Calcule un score de pertinence Flask [0, 1] basé sur les mots-clés.
        """
        query_lower = query.lower()
        words       = set(re.findall(r'\b\w+\b', query_lower))
 
        positive_hits = len(words & self.FLASK_POSITIVE_KEYWORDS)
        negative_hits = len(words & self.OUT_OF_SCOPE_KEYWORDS)
 
        if positive_hits > 0:
            return min(1.0, 0.5 + positive_hits * 0.2)
        elif negative_hits > 0:
            return max(0.0, 0.4 - negative_hits * 0.2)
        else:
            return 0.5  # ambigu — laisser passer
 
    def run(self, query: str) -> dict:
        """
        Évalue si la question est dans le périmètre Flask.
 
        Retourne :
          {
            "in_scope":   bool,
            "confidence": float,      # [0, 1]
            "reason":     str,        # explication de la décision
            "response":   str | None, # message à retourner si hors-scope
            "action":     str         # "proceed" | "reject" | "warn"
          }
        """
        # Vérification 1 : autre framework détecté
        other_fw = self._detect_other_framework(query)
        if other_fw:
            return {
                "in_scope":   False,
                "confidence": 0.95,
                "reason":     f"Question concernant {other_fw}, pas Flask",
                "response":   self.OUT_OF_SCOPE_RESPONSES["other_framework"].format(framework=other_fw),
                "action":     "reject",
            }
 
        # Vérification 2 : score de mots-clés Flask
        kw_score = self._compute_flask_relevance_score(query)
 
        if kw_score < 0.25:
            return {
                "in_scope":   False,
                "confidence": 0.80,
                "reason":     "Question non technique ou sans rapport avec Flask",
                "response":   self.OUT_OF_SCOPE_RESPONSES["non_technical"],
                "action":     "reject",
            }
 
        # Vérification 3 : test de retrieval rapide (top-1)
        quick_results = self.retriever.search(query, top_k=1, min_score=0.15)
 
        if not quick_results:
            return {
                "in_scope":   False,
                "confidence": 0.70,
                "reason":     "Aucun document Flask pertinent trouvé",
                "response":   self.OUT_OF_SCOPE_RESPONSES["low_relevance"],
                "action":     "reject",
            }
 
        top_score = quick_results[0].score
 
        if top_score < self.threshold:
            return {
                "in_scope":   True,   # laisser passer mais avec avertissement
                "confidence": 0.55,
                "reason":     f"Score faible ({top_score:.3f}) — résultat possible mais incertain",
                "response":   None,
                "action":     "warn",
            }
 
        return {
            "in_scope":   True,
            "confidence": min(1.0, kw_score + top_score) / 2,
            "reason":     f"Question Flask valide (score={top_score:.3f})",
            "response":   None,
            "action":     "proceed",
        }

**ORCHESTRATEUR — Pipeline RAG Agentique**

In [79]:
class AgenticRAGPipeline:
    """
    Orchestrateur du pipeline RAG agentique complet.
 
    Ce pipeline intègre les trois agents dans un workflow cohérent :
 
    Question
      ↓
    [Agent Routing] → hors-scope → Réponse de rejet
      ↓ in-scope
    [Retrieval initial]
      ↓
    [Agent Reformulation] → si score faible → reformulation + re-retrieval
      ↓
    [Agent Validation Code] → vérification syntaxique des blocs de code
      ↓
    [Construction du prompt augmenté]
      ↓
    [LLM] → Réponse sourcée
 
    La valeur ajoutée par rapport à un RAG classique :
      ✅ Moins d'hallucinations (routing + validation)
      ✅ Meilleur retrieval (reformulation)
      ✅ Code plus fiable (validation syntaxique)
      ✅ Meilleure UX (messages clairs hors-scope)
    """
 
    SYSTEM_PROMPT = """Tu es un assistant technique expert en Flask, spécialisé dans la documentation officielle Flask 3.0.x.
 
Règles strictes à respecter absolument :
1. Tu dois répondre UNIQUEMENT en te basant sur les extraits de documentation fournis dans le contexte.
2. Si l'information recherchée n'est pas présente dans les extraits fournis, réponds exactement : "Je ne trouve pas cette information dans la documentation Flask 3.0.x fournie."
3. À la fin de chaque réponse, indique la source utilisée au format : Source : [Titre de la page] — [Section] — [URL]
4. Si des extraits de code sont disponibles dans le contexte, intègre-les dans ta réponse en les formatant correctement.
5. Réponds en français si la question est posée en français, en anglais si la question est en anglais.
6. Ne génère aucune information, syntaxe ou comportement Flask qui ne soit pas explicitement mentionné dans les extraits.
7. Si le code fourni a été corrigé automatiquement, mentionne-le à l'utilisateur."""
 
    def __init__(self, retriever: BaseRetriever,
                 routing_agent: RoutingAgent,
                 reformulation_agent: QueryReformulationAgent,
                 validation_agent: CodeValidationAgent,
                 max_context_chars: int = 3000):
 
        self.retriever           = retriever
        self.routing_agent       = routing_agent
        self.reformulation_agent = reformulation_agent
        self.validation_agent    = validation_agent
        self.max_context_chars   = max_context_chars
 
    def _build_prompt(self, question: str, results: list, metadata: dict) -> str:
        """Construit le prompt augmenté final."""
        if not results:
            context = "Aucun extrait de documentation pertinent trouvé."
        else:
            blocks      = []
            total_chars = 0
            for i, r in enumerate(results, 1):
                block = f"--- Extrait {i} ---\n{r.to_context_block()}"
                if total_chars + len(block) > self.max_context_chars:
                    break
                blocks.append(block)
                total_chars += len(block)
            context = "\n\n".join(blocks)
 
        # Ajouter des notes agentiques si pertinent
        agent_notes = []
        if metadata.get("reformulated"):
            agent_notes.append(
                f"[Note : la question a été reformulée de '{metadata['original_query']}' "
                f"vers '{metadata['used_query']}' pour améliorer la recherche]"
            )
        if metadata.get("code_warnings"):
            agent_notes.append(
                "[Note : certains blocs de code ont des avertissements de compatibilité Flask 3.0.x]"
            )
        if metadata.get("routing_action") == "warn":
            agent_notes.append(
                "[Note : la pertinence de la question par rapport à Flask est incertaine — "
                "les résultats peuvent être imprécis]"
            )
 
        notes_section = ("\n\n=== NOTES SYSTÈME ===\n" + "\n".join(agent_notes)) if agent_notes else ""
 
        return (
            f"{self.SYSTEM_PROMPT}\n\n"
            f"=== DOCUMENTATION FLASK 3.0.x (extraits pertinents) ===\n\n"
            f"{context}"
            f"{notes_section}\n\n"
            f"=== QUESTION ===\n{question}\n\n"
            f"=== RÉPONSE ==="
        )
 
    def run(self, question: str, top_k: int = 4) -> dict:
        """
        Exécute le pipeline RAG agentique complet.
 
        Retourne un dictionnaire complet avec la réponse, les sources,
        les métadonnées agentiques et les statistiques d'exécution.
        """
        t_start  = time.perf_counter()
        metadata = {"question": question}
 
        print(f"\n{'='*60}")
        print(f"  Question : {question[:70]}...")
        print(f"{'='*60}")
 
        # ── ÉTAPE 1 : Routing ─────────────────────────────────────────────
        print("\n[1/4] Agent Routing...")
        routing_result = self.routing_agent.run(question)
        metadata["routing_action"]     = routing_result["action"]
        metadata["routing_confidence"] = routing_result["confidence"]
 
        print(f"  → Décision : {routing_result['action'].upper()} "
              f"(confiance={routing_result['confidence']:.2f})")
        print(f"  → Raison : {routing_result['reason']}")
 
        if not routing_result["in_scope"]:
            elapsed = time.perf_counter() - t_start
            return {
                "answer":           routing_result["response"],
                "sources":          [],
                "chunks_used":      0,
                "in_scope":         False,
                "reformulated":     False,
                "original_question":question,
                "used_question":    question,
                "validation_report":None,
                "prompt":           None,
                "elapsed_s":        round(elapsed, 2),
                "agent_metadata":   metadata,
            }
 
        # ── ÉTAPE 2 : Retrieval initial ───────────────────────────────────
        print("\n[2/4] Retrieval initial...")
        initial_results = self.retriever.search(question, top_k=top_k, min_score=0.20)
        print(f"  → {len(initial_results)} chunks trouvés")
        if initial_results:
            print(f"  → Meilleur score : {initial_results[0].score:.4f}")
 
        # ── ÉTAPE 3 : Reformulation si nécessaire ────────────────────────
        print("\n[3/4] Agent Reformulation...")
        reform_result = self.reformulation_agent.run(
            question, initial_results, top_k=top_k
        )
        final_results   = reform_result["results"]
        used_question   = reform_result["used_query"]
        metadata.update({
            "reformulated":   reform_result["reformulated"],
            "original_query": reform_result["original_query"],
            "used_query":     used_question,
            "reform_attempts":reform_result["attempts"],
            "best_score":     reform_result["best_score"],
        })
 
        if reform_result["reformulated"]:
            print(f"  → Reformulation effectuée en {reform_result['attempts']} tentatives")
            print(f"  → Score amélioré à {reform_result['best_score']:.4f}")
        else:
            print(f"  → Pas de reformulation nécessaire (score={reform_result['best_score']:.4f})")
 
        # ── ÉTAPE 4 : Validation du code ──────────────────────────────────
        print("\n[4/4] Agent Validation Code...")
        validation_report   = self.validation_agent.run(final_results)
        metadata["code_warnings"] = len(validation_report["blocks_with_warnings"]) > 0
 
        print(f"  → {validation_report['total_blocks']} blocs de code analysés")
        print(f"  → {validation_report['valid_blocks']} valides")
        if validation_report["blocks_with_errors"]:
            print(f"  → ⚠️ {len(validation_report['blocks_with_errors'])} erreurs détectées et corrigées")
 
        # ── Construction du prompt ────────────────────────────────────────
        prompt = self._build_prompt(used_question, final_results, metadata)
 
        # ── Préparation des sources ───────────────────────────────────────
        sources = [
            {
                "page_title": r.page_title,
                "section":    r.section,
                "source_url": r.source_url,
                "score":      r.score,
                "has_code":   r.has_code,
            }
            for r in final_results
        ]
 
        elapsed = time.perf_counter() - t_start
 
        print(f"\n✅ Pipeline terminé en {elapsed:.2f}s")
        print(f"   Chunks utilisés : {len(final_results)}")
        print(f"   Reformulé      : {metadata['reformulated']}")
        print(f"   Score final    : {metadata.get('best_score', 0):.4f}")
 
        return {
            "answer":           None,         # rempli par le LLM (Ollama local ou API)
            "prompt":           prompt,        # prompt prêt à envoyer au LLM
            "sources":          sources,
            "chunks_used":      len(final_results),
            "in_scope":         True,
            "reformulated":     metadata["reformulated"],
            "original_question":question,
            "used_question":    used_question,
            "validation_report":validation_report,
            "elapsed_s":        round(elapsed, 2),
            "agent_metadata":   metadata,
        }
 

**Instanciation et test du pipeline agentique**

In [80]:
# Instanciation des agents
routing_agent       = RoutingAgent(retriever, low_relevance_threshold=0.30)
reformulation_agent = QueryReformulationAgent(retriever, min_score=0.35)
validation_agent    = CodeValidationAgent()
 
# Pipeline complet
agentic_pipeline = AgenticRAGPipeline(
    retriever           = retriever,
    routing_agent       = routing_agent,
    reformulation_agent = reformulation_agent,
    validation_agent    = validation_agent,
    max_context_chars   = 3000,
)
 
print("✅ Pipeline agentique instancié avec succès")
print("\nComposants :")
print("  ✅ BaseRetriever          (ChromaDB + all-MiniLM-L6-v2)")
print("  ✅ RoutingAgent           (détection hors-scope)")
print("  ✅ QueryReformulationAgent(amélioration retrieval)")
print("  ✅ CodeValidationAgent    (validation syntaxique Python)")
print("  ✅ AgenticRAGPipeline     (orchestrateur complet)")
 

✅ Pipeline agentique instancié avec succès

Composants :
  ✅ BaseRetriever          (ChromaDB + all-MiniLM-L6-v2)
  ✅ RoutingAgent           (détection hors-scope)
  ✅ QueryReformulationAgent(amélioration retrieval)
  ✅ CodeValidationAgent    (validation syntaxique Python)
  ✅ AgenticRAGPipeline     (orchestrateur complet)


**Tests complets du pipeline**

In [81]:
test_questions = [
    # Questions in-scope Flask
    ("Flask route", "Comment créer une route Flask avec un paramètre dynamique ?"),
    ("Flask debug", "How to enable debug mode in Flask?"),
    ("Flask blueprint","Comment utiliser les blueprints Flask pour organiser une application ?"),
    # Question avec reformulation attendue
    ("Reformulation",  "comment faire une page web avec python flask"),
    # Questions hors-scope
    ("Hors-scope 1",   "Comment créer un modèle de machine learning avec PyTorch ?"),
    ("Hors-scope 2",   "Quelle est la recette de la tarte aux pommes ?"),
    ("Autre framework","Comment définir des routes dans Django ?"),
]
 
results_summary = []
 
for category, question in test_questions:
    print(f"\n{'#'*60}")
    print(f"  Test [{category}]")
    print(f"{'#'*60}")
 
    result = agentic_pipeline.run(question, top_k=3)
 
    results_summary.append({
        "category":    category,
        "question":    question,
        "in_scope":    result["in_scope"],
        "reformulated":result["reformulated"],
        "chunks":      result["chunks_used"],
        "score":       result["agent_metadata"].get("best_score", 0),
        "elapsed_s":   result["elapsed_s"],
    })
 
    if not result["in_scope"]:
        print(f"\n📢 Réponse hors-scope :")
        print(f"   {result['answer'][:200]}...")
    else:
        print(f"\n📋 Prompt prêt (premiers 300 chars) :")
        print(f"   {result['prompt'][:300]}...")
        print(f"\n📚 Sources ({len(result['sources'])}) :")
        for src in result["sources"][:3]:
            print(f"   [{src['score']:.3f}] {src['page_title']} — {src['section']}")
 


############################################################
  Test [Flask route]
############################################################

  Question : Comment créer une route Flask avec un paramètre dynamique ?...

[1/4] Agent Routing...
  → Décision : PROCEED (confiance=0.50)
  → Raison : Question Flask valide (score=0.734)

[2/4] Retrieval initial...
  → 3 chunks trouvés
  → Meilleur score : 0.7338

[3/4] Agent Reformulation...
  → Pas de reformulation nécessaire (score=0.7338)

[4/4] Agent Validation Code...
  → 0 blocs de code analysés
  → 0 valides

✅ Pipeline terminé en 0.05s
   Chunks utilisés : 3
   Reformulé      : False
   Score final    : 0.7338

📋 Prompt prêt (premiers 300 chars) :
   Tu es un assistant technique expert en Flask, spécialisé dans la documentation officielle Flask 3.0.x.
 
Règles strictes à respecter absolument :
1. Tu dois répondre UNIQUEMENT en te basant sur les extraits de documentation fournis dans le contexte.
2. Si l'information recherchée n'est 

**Rapport de performance**

In [82]:
print(f"\n{'='*65}")
print(f"  RAPPORT DE PERFORMANCE — MODULE 4 AGENTIQUE")
print(f"{'='*65}")
print(f"  {'Catégorie':<20} {'In-scope':<10} {'Réformulé':<12} {'Chunks':<8} {'Score':<8} {'Temps'}")
print(f"  {'-'*60}")
for r in results_summary:
    print(f"  {r['category']:<20} "
          f"{'✅' if r['in_scope'] else '❌':<10} "
          f"{'✅' if r['reformulated'] else '—':<12} "
          f"{r['chunks']:<8} "
          f"{r['score']:.3f}{'':3} "
          f"{r['elapsed_s']:.2f}s")
 
in_scope_count = sum(1 for r in results_summary if r["in_scope"])
reformulated   = sum(1 for r in results_summary if r["reformulated"])
avg_time       = sum(r["elapsed_s"] for r in results_summary) / len(results_summary)
 
print(f"\n  Résumé :")
print(f"  ├─ Questions testées     : {len(results_summary)}")
print(f"  ├─ Questions in-scope    : {in_scope_count}")
print(f"  ├─ Questions reformulées : {reformulated}")
print(f"  └─ Temps moyen pipeline  : {avg_time:.2f}s")
print(f"{'='*65}")
 


  RAPPORT DE PERFORMANCE — MODULE 4 AGENTIQUE
  Catégorie            In-scope   Réformulé    Chunks   Score    Temps
  ------------------------------------------------------------
  Flask route          ✅          —            3        0.734    0.05s
  Flask debug          ✅          —            3        0.876    0.11s
  Flask blueprint      ✅          —            3        0.846    0.02s
  Reformulation        ✅          —            3        0.775    0.02s
  Hors-scope 1         ❌          —            0        0.000    0.00s
  Hors-scope 2         ❌          —            0        0.000    0.00s
  Autre framework      ❌          —            0        0.000    0.00s

  Résumé :
  ├─ Questions testées     : 7
  ├─ Questions in-scope    : 4
  ├─ Questions reformulées : 0
  └─ Temps moyen pipeline  : 0.03s


**Export pour usage local**

In [83]:
import zipfile, shutil
 
# Sauvegarder ChromaDB + agents dans un ZIP pour télécharger
OUTPUT_DIR = Path("/kaggle/working/chatbot_export")
OUTPUT_DIR.mkdir(exist_ok=True)
 
# Copier ChromaDB
shutil.copytree(CHROMA_DIR, OUTPUT_DIR / "vector_db", dirs_exist_ok=True)
 
# Sauvegarder le rapport
with open(OUTPUT_DIR / "performance_report.json", "w") as f:
    json.dump(results_summary, f, indent=2, ensure_ascii=False)
 
# Créer le ZIP final
zip_path = "/kaggle/working/chatbot_rag_export.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file in OUTPUT_DIR.rglob("*"):
        if file.is_file():
            zf.write(file, file.relative_to(OUTPUT_DIR))
 
print(f"\n✅ Export créé : {zip_path}")
print(f"   → Téléchargez ce fichier depuis Kaggle Output")
print(f"   → Extrayez vector_db/ dans votre dossier chatbot_doc_flask/")
print(f"   → Lancez : streamlit run app.py")


✅ Export créé : /kaggle/working/chatbot_rag_export.zip
   → Téléchargez ce fichier depuis Kaggle Output
   → Extrayez vector_db/ dans votre dossier chatbot_doc_flask/
   → Lancez : streamlit run app.py
